# Bank Transaction Fraud Detection
## Feature Engineering & Data Preparation

**Group B, Member 1 — Week 3 Deliverable**

This notebook prepares the cleaned dataset (per Group A's Week 2 EDA handoff notes) for model training: dropping flagged columns, engineering time features, encoding categoricals, scaling numerics, splitting train/test, and addressing the 5.04% class imbalance.

## 1. Imports & Load Data

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
import joblib

pd.set_option('display.max_columns', None)

df = pd.read_csv('Bank_Transaction_Fraud_Detection.csv')
print('Raw shape:', df.shape)
df.head()

## 2. Drop Flagged Columns

Per Group A's handoff notes:
- `Transaction_Currency` — constant (single value 'INR')
- `Customer_ID`, `Customer_Name`, `Customer_Contact`, `Customer_Email` — PII / unique identifiers
- `Transaction_ID`, `Merchant_ID` — unique identifiers, no generalizable signal
- `Bank_Branch`, `Transaction_Location` — verified exact duplicates of City/State (0 mismatches across 200,000 rows)

In [ ]:
drop_cols = [
    'Transaction_Currency',
    'Customer_ID', 'Customer_Name', 'Customer_Contact', 'Customer_Email',
    'Transaction_ID', 'Merchant_ID',
    'Bank_Branch', 'Transaction_Location'
]
df = df.drop(columns=drop_cols)
print('After dropping flagged columns:', df.shape)

## 3. Engineer Time Features

`Transaction_Date` and `Transaction_Time` are split into numeric components (`Transaction_Day`, `Transaction_Month`, `Transaction_DayOfWeek`, `Transaction_Hour`, `Transaction_Minute`, `Transaction_Second`) as recommended in the EDA handoff, since raw features showed no individual signal — interaction/time features are the next avenue to explore.

In [ ]:
df['Transaction_Date'] = pd.to_datetime(df['Transaction_Date'], format='%d-%m-%Y')
time_parts = df['Transaction_Time'].str.split(':', expand=True).astype(int)

df['Transaction_Day'] = df['Transaction_Date'].dt.day
df['Transaction_Month'] = df['Transaction_Date'].dt.month
df['Transaction_DayOfWeek'] = df['Transaction_Date'].dt.dayofweek  # 0=Monday
df['Transaction_Hour'] = time_parts[0]
df['Transaction_Minute'] = time_parts[1]
df['Transaction_Second'] = time_parts[2]

df = df.drop(columns=['Transaction_Date', 'Transaction_Time'])
print('After engineering time features:', df.shape)
df.filter(like='Transaction_').head()

## 4. Encode Categorical Variables

Label encoding is applied to the remaining categorical columns. Encoders are saved (`label_encoders.pkl`) so the same mapping can be reused at inference time in the Streamlit app.

In [ ]:
categorical_cols = ['Gender', 'State', 'City', 'Account_Type', 'Transaction_Type',
                     'Merchant_Category', 'Transaction_Device', 'Device_Type',
                     'Transaction_Description']

encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    encoders[col] = le

df[categorical_cols].head()

## 5. Train/Test Split

Performed **before** scaling and resampling to prevent data leakage. Stratified on `Is_Fraud` to preserve the 5.04% fraud rate in both splits.

In [ ]:
X = df.drop(columns=['Is_Fraud'])
y = df['Is_Fraud']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Train shape:', X_train.shape, '| Test shape:', X_test.shape)
print(f'Train fraud rate: {y_train.mean():.4f} | Test fraud rate: {y_test.mean():.4f}')

## 6. Scale Numeric Features

`StandardScaler` is fit on the **training set only** and applied to both splits to avoid leakage. Scaler is saved (`scaler.pkl`) for reuse in deployment.

In [ ]:
numeric_cols = ['Age', 'Transaction_Amount', 'Account_Balance',
                 'Transaction_Day', 'Transaction_Month', 'Transaction_DayOfWeek',
                 'Transaction_Hour', 'Transaction_Minute', 'Transaction_Second']

scaler = StandardScaler()
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

X_train[numeric_cols].describe().loc[['mean','std']]

## 7. Address Class Imbalance — 5:1 Oversampling

The EDA flagged a 94.96% / 5.04% class split. `imbalanced-learn` (SMOTE) was not available in this environment, so we use the same workaround applied in the earlier solo CRISP-DM run of this project: `sklearn.utils.resample` to oversample the minority (fraud) class on the **training set only**, targeting roughly a 5:1 majority:minority ratio rather than full 1:1 balance, to avoid excessive synthetic duplication while still giving the minority class meaningfully more weight during training.

In [ ]:
train_df = X_train.copy()
train_df['Is_Fraud'] = y_train.values

majority = train_df[train_df.Is_Fraud == 0]
minority = train_df[train_df.Is_Fraud == 1]

target_minority_size = len(majority) // 5
minority_upsampled = resample(
    minority, replace=True, n_samples=target_minority_size, random_state=42
)

train_balanced = pd.concat([majority, minority_upsampled]).sample(frac=1, random_state=42).reset_index(drop=True)

print('Balanced train shape:', train_balanced.shape)
print(f'Balanced train fraud rate: {train_balanced.Is_Fraud.mean():.4f}')

## 8. Save Processed Datasets & Artifacts

- `train_processed.csv` — balanced, scaled, encoded training set (for Group B Member 2 — Model Development)
- `test_processed.csv` — held-out, untouched-ratio test set (for Group C — Model Evaluation)
- `scaler.pkl`, `label_encoders.pkl` — fitted transformers for consistent reuse in Streamlit deployment

In [ ]:
test_df = X_test.copy()
test_df['Is_Fraud'] = y_test.values

train_balanced.to_csv('train_processed.csv', index=False)
test_df.to_csv('test_processed.csv', index=False)
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(encoders, 'label_encoders.pkl')

print('Saved: train_processed.csv, test_processed.csv, scaler.pkl, label_encoders.pkl')

## 9. Summary & Handoff to Group B Member 2 (Model Development)

| Item | Value |
|---|---|
| Raw shape | 200,000 rows × 24 columns |
| Columns dropped | 9 (constant / PII / redundant) |
| Time features engineered | 6 |
| Final feature count | 18 |
| Train shape (balanced) | 182,316 × 19 |
| Train fraud rate (balanced) | 16.67% (~5:1) |
| Test shape | 40,000 × 19 |
| Test fraud rate (untouched) | 5.05% |

**Next step (Group B Member 2):** train Logistic Regression, Decision Tree, Random Forest, and XGBoost/Gradient Boosting on `train_processed.csv`, evaluate on the untouched `test_processed.csv`.